In [ ]:
import pandas as pd
import numpy as np
import warnings
import random
import os
import re

# Setting seeds to maintain logic
my_seed = 42
def fix_seeds(s):
    random.seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    np.random.seed(s)

fix_seeds(my_seed)
warnings.filterwarnings('ignore')

# Loading like the Regression structure
df_dev = pd.read_csv("development.csv")
df_evaluation = pd.read_csv("evaluation.csv") 


In [2]:
df_dev.head()

,Id,source,title,article,page_rank,timestamp,label
0,0,AllAfrica.com,OPEC Boosts Nigeria&#39;s Oil Revenue By .82m Bpd,THE Organisation of Petroleum Exporting Countr...,5,2004-09-16 22:39:53,5
1,1,Xinhua,Yearender: Mideast peace roadmap reaches dead-...,Looking back at the major events that took pla...,5,2004-12-17 19:01:14,0
2,2,Yahoo,Battleground Dispatches for Oct. 5 \\n (CQP...,CQPolitics.com - Here are today's Battleground...,5,2006-10-05 18:42:29,0
3,3,BBC,Air best to resuscitate newborns,Air rather than oxygen should be used to resus...,5,0000-00-00 00:00:00,0
4,4,Yahoo,High tech German train crash kills at least on...,"<p><a href=""http://us.rd.yahoo.com/dailynews/r...",5,2006-09-22 17:28:57,0


In [3]:
# 1. Convert timestamp FIRST while labels are still in the same dataframe
df_dev["timestamp"] = pd.to_datetime(df_dev["timestamp"], errors="coerce")

# 2. Sort the ENTIRE dataframe (Features + Labels move together)
df_sorted = df_dev.sort_values("timestamp", na_position="first").reset_index(drop=True)

# 3. Determine split point
split_idx = int(0.70 * len(df_sorted))

# 4. Split the sorted dataframe
train_set = df_sorted.iloc[:split_idx].copy()
val_set = df_sorted.iloc[split_idx:].copy()

# 5. Separate features (X) and targets (y) for both sets
y_train = train_set["label"]
df_train = train_set.drop(columns="label")

y_val = val_set["label"]
df_val = val_set.drop(columns="label")

# 6. Create the 'text' column for the pipeline (Cleaning and Combining)
df_train['text'] = df_train['title'].fillna('') + " " + df_train['article'].fillna('')
df_val['text'] = df_val['title'].fillna('') + " " + df_val['article'].fillna('')

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
# --- Helper Functions (Revised for performance) ---
top_sources = df_train["source"].value_counts().nlargest(250).index.tolist()

def group_src(x):
    # This must return a 2D shape (Vertical Column) for OneHotEncoder
    res = x.apply(lambda val: val if val in top_sources else "Other")
    return res.values.reshape(-1, 1) # Reshaping to 2D solves the ValueError

def clean_text(df_in):
    # This must return a 1D Sequence of strings for Tfidf
    combined = df_in.iloc[:, 0].fillna("") + " " + df_in.iloc[:, 1].fillna("")
    return combined.apply(lambda x: re.sub(r"\s+", " ", str(x).replace(r'\N', '')).strip()).tolist()

min_year = df_train["timestamp"].dt.year.min()

def get_time_feats(df_in):
    temp = pd.DataFrame(index=df_in.index)
    # Using a slightly faster approach for time
    ts = pd.to_datetime(df_in.iloc[:, 0]) 
    h = ts.dt.hour
    temp['seg'] = h.apply(lambda x: -1 if pd.isna(x) else 0 if x < 7 else 1 if x < 12 else 2 if x < 18 else 3)
    temp['mo_yr'] = (ts.dt.year - min_year) * 12 + ts.dt.month
    temp['dow'] = ts.dt.dayofweek
    return temp.fillna(-1).astype(int)

# --- The Pipeline ---
preproc = ColumnTransformer(
    [
        # Notice we use 'ngram_range' here—crucial for that 0.74 score!
        ("text", make_pipeline(
            FunctionTransformer(clean_text, validate=False), 
            TfidfVectorizer(ngram_range=(1, 3), max_features=25000, sublinear_tf=True, stop_words="english")
        ), ["title", "article"]),
        
        ("cat", make_pipeline(
            FunctionTransformer(group_src, validate=False),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        ), "source"),
        
        ("cont", make_pipeline(
            SimpleImputer(strategy="mean"), 
            MinMaxScaler()
        ), ["page_rank"]),
        
        ("time", make_pipeline(
            FunctionTransformer(get_time_feats, validate=False),
            StandardScaler()
        ), ["timestamp"])
    ],
    remainder="drop"
)

In [6]:
X_train_pp = preproc.fit_transform(df_train)
X_val_pp = preproc.transform(df_val)

In [7]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# 1. Initialize the model (Regression style: direct instantiation)
# We use class_weight='balanced' because news categories are often imbalanced
clf = LinearSVC(C=0.1, class_weight="balanced", random_state=my_seed, dual=False)

# 2. Fit the model (Using the _pp training data)
clf.fit(X_train_pp, y_train)

# 3. Predict on validation (Using the _pp validation data)
y_val_predicted = clf.predict(X_val_pp)

# 4. Print the result
score = f1_score(y_val, y_val_predicted, average="macro")
print(f"Validation Macro F1 Score: {score:.4f}")

Validation Macro F1 Score: 0.7446


In [10]:
from scipy.sparse import vstack 
import pandas as pd

# 1. Re-stack the full processed development set (Train + Val)
X_full_pp = vstack([X_train_pp, X_val_pp])
y_full = pd.concat([y_train, y_val])

# 2. Re-train the model on the entire dataset
print("Training final model on full development set...")
clf_final = LinearSVC(C=0.1, class_weight="balanced", random_state=my_seed, dual=False)
clf_final.fit(X_full_pp, y_full)

# 3. Process the Evaluation set (Test data)
# FIX: Coerce the bad timestamps in the evaluation set to NaT, just like you did for the training set!
df_evaluation["timestamp"] = pd.to_datetime(df_evaluation["timestamp"], errors="coerce")

# Now the pipeline will transform it without crashing
X_test_pp = preproc.transform(df_evaluation)

# 4. Generate Predictions
y_test_pred = clf_final.predict(X_test_pp)

# 5. Create Submission File
submission = pd.DataFrame({
    "Id": df_evaluation["Id"], 
    "label": y_test_pred
})

submission.to_csv("submission.csv", index=False)
print("Submission file saved as 'submission.csv'!")

Training final model on full development set...
Submission file saved as 'submission.csv'!
